# Clustering and recommender: step by step

This notebook walks through the two ML components of the Learning Path Recommender.
Run the cells top to bottom (select the `.venv` kernel). Use it as the **technical explanation** for your portfolio.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.metrics import adjusted_rand_score

from src.config import DEMO_PROFILES, TOPICS, TOPIC_LABELS
from src.data_generation import generate_students
from src.clustering import train_cluster_model, assign_cluster
from src.recommender import load_resources, recommend
from src.visuals import cluster_scatter, k_selection_chart, radar_chart

## 1. Synthetic learners
There is no public dataset of quiz skill vectors, so we generate 300 learners from 5 designed archetypes plus noise.
**Say this clearly in your portfolio.**

In [ ]:
students = generate_students(n=300)
students.head()

In [ ]:
students[TOPICS].describe().round(2)

## 2. Choosing k for K-Means
We try k = 3 to 7 and compare the **elbow** (inertia) and the **silhouette score** (higher = better separated clusters).

In [ ]:
model = train_cluster_model(students)
display(model.k_scores.round(3))
k_selection_chart(model.k_scores, model.k)

## 3. Cluster profiles
Each centroid (average skill vector) is turned into a human-readable archetype name, focus and advice.

In [ ]:
profiles = pd.DataFrame(
    [{"cluster": i, "name": p.name, "focus": p.focus, **dict(zip(TOPICS, p.centroid))}
     for i, p in model.profiles.items()]
)
profiles

In [ ]:
cluster_scatter(model, students)

## 4. Sanity check against the hidden archetypes
The `true_archetype` column was never given to K-Means. The **Adjusted Rand Index** measures how well the clusters match it
(1.0 = perfect). Remember this is synthetic data, so it validates the pipeline, not real-world learner types.

In [ ]:
print("k =", model.k)
print("silhouette =", round(model.silhouette, 3))
print("adjusted Rand index =", round(adjusted_rand_score(students["true_archetype"], model.labels), 3))

## 5. The recommender
Scoring (see `docs/architecture.md`): cosine similarity between the learner's **gap vector** and each resource,
plus the size of the main gap, difficulty fit, prerequisite and time-budget adjustments.

In [ ]:
resources = load_resources()
resources["topic"].value_counts()

In [ ]:
name = "Practical coder, light on theory"   # try the other keys of DEMO_PROFILES too
skills = DEMO_PROFILES[name]
print("Learner type:", assign_cluster(model, skills)[1].name)
radar_chart(skills)

In [ ]:
rec = recommend(skills, resources, hours_per_week=6)
rec[["order", "id", "title", "topic", "difficulty", "hours", "score", "reason"]]

## 6. Test on hand-made profiles
Check each path makes sense. Edit the dictionary and re-run to explore.

In [ ]:
tests = {
    "Strong Python, no ML": dict(python=1.0, data=0.67, prompting=0.67, ml=0.0, evaluation=0.0, ethics=0.67),
    "No coding, strong prompting": dict(python=0.0, data=0.0, prompting=1.0, ml=0.33, evaluation=0.0, ethics=0.33),
}
for label, s in tests.items():
    r = recommend(s, resources, hours_per_week=6)
    print(label, "->", ", ".join(r["id"]))

## 7. My reflections
- What worked well:
- What surprised me:
- What I would improve (real data, collaborative filtering, adaptive quiz):